In [5]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np

from hloc import (
    extract_features,
    match_features,
    reconstruction,
    visualization,
    pairs_from_retrieval,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Setup
In this notebook, we will run SfM reconstruction from scratch on a set of images. First, we define some paths.

In [6]:
images = Path("datasets/ZEDX_Mini/09/mav0/cam0/data/")

outputs = Path("outputs/zedx_mini/09")
sfm_pairs = outputs / "pairs-megaloc.txt"
sfm_dir = outputs / "sfm_superpoint+lightglue"

retrieval_conf = extract_features.confs["megaloc"]
feature_conf = extract_features.confs["superpoint_max"]
matcher_conf = match_features.confs["superpoint+lightglue"]

kf_poses = "/home/hapq/Documents/SLAM_Testings/dso_maps/sp_gl_vmo_office_768x480_kf_poses.txt"
kf_filenames = []
pose_priors = {}
kf_name_to_id = {}
with open(kf_poses, "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        filename = parts[1]
        tx = float(parts[2])
        ty = float(parts[3])
        tz = float(parts[4])
        qx = float(parts[5])
        qy = float(parts[6])
        qz = float(parts[7])
        qw = float(parts[8])
        qvec = np.array([qw, qx, qy, qz], dtype=np.float64)
        tvec = np.array([tx, ty, tz], dtype=np.float64)
        kf_filenames.append(filename)
        pose_priors[filename] = (qvec, tvec)
        original_id = int(parts[0])
        kf_name_to_id[filename] = original_id
kf_filenames_sub = kf_filenames[::2]
pose_priors_sub = {
    name: pose_priors[name]
    for name in kf_filenames_sub
}

## Find image pairs via image retrieval
We extract global descriptors with NetVLAD and find for each image the most similar ones. For smaller dataset we can instead use exhaustive matching via `hloc/pairs_from_exhaustive.py`, which would find $\frac{n(n-1)}{2}$ images pairs.

In [10]:
retrieval_path = extract_features.main(retrieval_conf, images, outputs, image_list=kf_filenames_sub)
pairs_from_retrieval.main(retrieval_path, sfm_pairs, num_matched=5)

[2026/03/03 08:56:52 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'megaloc'},
 'output': 'global-feats-megaloc',
 'preprocessing': {'resize_max': 1024}}
[2026/03/03 08:56:53 hloc INFO] Skipping the extraction.
[2026/03/03 08:56:53 hloc INFO] Extracting image pairs from a retrieval database.
[2026/03/03 08:56:54 hloc INFO] Found 12395 pairs.


## Extract and match local features

In [11]:
feature_path = extract_features.main(feature_conf, images, outputs, image_list=kf_filenames_sub)
match_path = match_features.main(
    matcher_conf, sfm_pairs, feature_conf["output"], outputs
)

[2026/03/03 08:57:01 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-rmax1600',
 'preprocessing': {'grayscale': True, 'resize_force': True, 'resize_max': 1600}}
[2026/03/03 08:57:02 hloc INFO] Skipping the extraction.
[2026/03/03 08:57:02 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
[2026/03/03 08:57:02 hloc INFO] Skipping the matching.


## 3D reconstruction
Run COLMAP on the features and matches.

In [12]:
opts = dict(camera_model='PINHOLE', camera_params=','.join(map(str, (367.411163, 367.411163, 482.177521, 297.930115))))
model = reconstruction.main(sfm_dir, images, sfm_pairs, feature_path, match_path, image_list=kf_filenames_sub, pose_priors=pose_priors_sub, image_options=opts, mapper_options=dict(use_prior_position=True))

[2026/03/03 08:57:11 hloc INFO] Writing COLMAP logs to outputs/zedx_mini/09/sfm_superpoint+lightglue/colmap.LOG.*
[2026/03/03 08:57:11 hloc WARNING] The database already exists, deleting it.
[2026/03/03 08:57:11 hloc INFO] Creating an empty database...
[2026/03/03 08:57:11 hloc INFO] Importing images into the database...
[2026/03/03 08:57:23 hloc INFO] Injecting STRONG pose priors (converted from Tcw)...
[2026/03/03 08:57:24 hloc INFO] Pose priors successfully written.
[2026/03/03 08:57:24 hloc INFO] Importing features into the database...
100%|██████████| 2479/2479 [00:01<00:00, 2260.87it/s]
[2026/03/03 08:57:25 hloc INFO] Importing matches into the database...
100%|██████████| 12395/12395 [00:04<00:00, 2795.96it/s]
[2026/03/03 08:57:29 hloc INFO] Performing geometric verification of the matches...
[2026/03/03 08:57:32 hloc INFO] Running 3D reconstruction...
Reconstruction 0: 100%|██████████| 2479/2479 [2:05:25<00:00,  3.04s/images, registered]    
[2026/03/03 11:03:00 hloc INFO] Reco

In [13]:
output_path = "optimized_poses.txt"
with open(output_path, "w") as f:
    f.write("#id filename tx ty tz qx qy qz qw\n")
    images = sorted(model.images.values(), key=lambda im: im.name)
    for image in images:
        if image.name not in kf_name_to_id:
            print(f"Warning: {image.name} not found in original kf file")
            continue
        original_id = kf_name_to_id[image.name]
        Tcw = image.cam_from_world()
        t = Tcw.translation
        q = Tcw.rotation.quat
        qw, qx, qy, qz = q
        f.write(
            f"{original_id} {image.name} "
            f"{t[0]:.15f} {t[1]:.15f} {t[2]:.15f} "
            f"{qx:.15f} {qy:.15f} {qz:.15f} {qw:.15f}\n"
        )
print("Saved optimized poses.")

Saved optimized poses.


## Visualization
We visualize some of the registered images, and color their keypoint by visibility, track length, or triangulated depth.

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="visibility", n=5)

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="track_length", n=5)

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="depth", n=5)